In [1]:
# Decision Tree 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the SmartThermostatDataset.csv dataset
df = pd.read_csv('SmartThermostatDataset.csv')

# Define feature columns (excluding 'Type' which is the label)
feature_columns = [
    'timestamp', 'source_ip', 'source_port', 'destination_ip', 'dst_port', 'protocol',
    'Family', 'Service', 'duration', 'source_bytes', 'destination_bytes', 'conn_state',
    'missed_bytes', 'source_packets', 'src_ip_bytes', 'destination_packets', 'dst_ip_bytes',
    'Temperature', 'Humidity', 'Status'
]

label_column = 'Type'  # 'Type' is the label

# Handle categorical features: Use LabelEncoder to convert categorical columns to numeric values
categorical_columns = ['protocol', 'Family', 'Service', 'conn_state', 'timestamp', 'source_ip', 'destination_ip', 'Status']
label_encoder = LabelEncoder()

for column in categorical_columns:
    if column in df.columns:
        df[column] = label_encoder.fit_transform(df[column].astype(str))

# Map string labels to numeric values for 'DOS' detection
df['Type'] = df['Type'].str.lower()  # Convert to lowercase

label_mapping = {
    'normal': 0,  # Non-DOS
    'mitm': 0,    # Non-DOS
    'dos': 1      # DOS
}

df['Label'] = df['Type'].map(label_mapping)

# Check for unmapped labels
if df['Label'].isna().sum() > 0:
    print("Unmapped labels found. Check your dataset!")
    print(df[df['Label'].isna()])
    raise ValueError("Unmapped labels detected.")

# Confirm the label distribution
print("Label distribution:")
print(df['Label'].value_counts())

# Drop any rows where feature columns contain non-numeric or missing data
df = df.dropna(subset=feature_columns)

# Convert all features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Extract features and labels
X = df[feature_columns].values
y = df['Label'].values.astype(np.float32)  # Use the mapped numeric 'Label' column

# Replace NaN values in each column with the column mean
for i in range(X.shape[1]):  # Iterate over columns
    if np.issubdtype(X[:, i].dtype, np.number):  # Ensure column is numeric
        column_mean = np.nanmean(X[:, i])  # Compute the mean of the column
        X[:, i] = np.where(np.isnan(X[:, i]), column_mean, X[:, i])  # Replace NaNs with the column mean

# Replace Infinite values with 0
X[np.isinf(X)] = 0

# Normalize the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize the Decision Tree Classifier
classifier = DecisionTreeClassifier(random_state=42)

# Train the classifier
classifier.fit(X_train, y_train)

# Make predictions
y_train_pred = classifier.predict(X_train)
y_test_pred = classifier.predict(X_test)

# Evaluate the model
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Detailed evaluation
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Label distribution:
0    3852
1     292
Name: Label, dtype: int64
Training Accuracy: 1.0000
Test Accuracy: 0.9940

Classification Report on Test Data:
              precision    recall  f1-score   support

         0.0       0.99      1.00      1.00       774
         1.0       0.98      0.93      0.95        55

    accuracy                           0.99       829
   macro avg       0.99      0.96      0.98       829
weighted avg       0.99      0.99      0.99       829


Confusion Matrix:
[[773   1]
 [  4  51]]


In [2]:
# Random Forest
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier  # Import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the SmartThermostatDataset.csv dataset
df = pd.read_csv('SmartThermostatDataset.csv')

# Define feature columns (excluding 'Type' which is the label)
feature_columns = [
    'timestamp', 'source_ip', 'source_port', 'destination_ip', 'dst_port', 'protocol',
    'Family', 'Service', 'duration', 'source_bytes', 'destination_bytes', 'conn_state',
    'missed_bytes', 'source_packets', 'src_ip_bytes', 'destination_packets', 'dst_ip_bytes',
    'Temperature', 'Humidity', 'Status'
]

label_column = 'Type'  # 'Type' is the label

# Handle categorical features: Use LabelEncoder to convert categorical columns to numeric values
categorical_columns = ['protocol', 'Family', 'Service', 'conn_state', 'timestamp', 'source_ip', 'destination_ip', 'Status']
label_encoder = LabelEncoder()

for column in categorical_columns:
    if column in df.columns:
        df[column] = label_encoder.fit_transform(df[column].astype(str))

# Map string labels to numeric values for 'DOS' detection
df['Type'] = df['Type'].str.lower()  # Convert to lowercase

label_mapping = {
    'normal': 0,  # Non-DOS
    'mitm': 0,    # Non-DOS
    'dos': 1      # DOS
}

df['Label'] = df['Type'].map(label_mapping)

# Check for unmapped labels
if df['Label'].isna().sum() > 0:
    print("Unmapped labels found. Check your dataset!")
    print(df[df['Label'].isna()])
    raise ValueError("Unmapped labels detected.")

# Confirm the label distribution
print("Label distribution:")
print(df['Label'].value_counts())

# Drop any rows where feature columns contain non-numeric or missing data
df = df.dropna(subset=feature_columns)

# Convert all features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Extract features and labels
X = df[feature_columns].values
y = df['Label'].values.astype(np.float32)  # Use the mapped numeric 'Label' column

# Replace NaN values in each column with the column mean
for i in range(X.shape[1]):  # Iterate over columns
    if np.issubdtype(X[:, i].dtype, np.number):  # Ensure column is numeric
        column_mean = np.nanmean(X[:, i])  # Compute the mean of the column
        X[:, i] = np.where(np.isnan(X[:, i]), column_mean, X[:, i])  # Replace NaNs with the column mean

# Replace Infinite values with 0
X[np.isinf(X)] = 0

# Normalize the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize the Random Forest Classifier
classifier = RandomForestClassifier(random_state=42, n_estimators=100)  # You can adjust n_estimators and other parameters

# Train the classifier
classifier.fit(X_train, y_train)

# Make predictions
y_train_pred = classifier.predict(X_train)
y_test_pred = classifier.predict(X_test)

# Evaluate the model
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Detailed evaluation
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))


Label distribution:
0    3852
1     292
Name: Label, dtype: int64
Training Accuracy: 1.0000
Test Accuracy: 0.9988

Classification Report on Test Data:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       774
         1.0       1.00      0.98      0.99        55

    accuracy                           1.00       829
   macro avg       1.00      0.99      1.00       829
weighted avg       1.00      1.00      1.00       829


Confusion Matrix:
[[774   0]
 [  1  54]]


In [3]:
#KNN 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

# Load the SmartThermostatDataset.csv dataset
df = pd.read_csv('SmartThermostatDataset.csv')

# Define feature columns (excluding 'Type' which is the label)
feature_columns = [
    'timestamp', 'source_ip', 'source_port', 'destination_ip', 'dst_port', 'protocol',
    'Family', 'Service', 'duration', 'source_bytes', 'destination_bytes', 'conn_state',
    'missed_bytes', 'source_packets', 'src_ip_bytes', 'destination_packets', 'dst_ip_bytes',
    'Temperature', 'Humidity', 'Status'
]

label_column = 'Type'  # 'Type' is the label

# Handle categorical features: Use LabelEncoder to convert categorical columns to numeric values
categorical_columns = ['protocol', 'Family', 'Service', 'conn_state', 'timestamp', 'source_ip', 'destination_ip', 'Status']
label_encoder = LabelEncoder()

for column in categorical_columns:
    if column in df.columns:
        df[column] = label_encoder.fit_transform(df[column].astype(str))

# Map string labels to numeric values for 'DOS' detection
df['Type'] = df['Type'].str.lower()  # Convert to lowercase

label_mapping = {
    'normal': 0,  # Non-DOS
    'mitm': 0,    # Non-DOS
    'dos': 1      # DOS
}

df['Label'] = df['Type'].map(label_mapping)

# Check for unmapped labels
if df['Label'].isna().sum() > 0:
    print("Unmapped labels found. Check your dataset!")
    print(df[df['Label'].isna()])
    raise ValueError("Unmapped labels detected.")

# Confirm the label distribution
print("Label distribution:")
print(df['Label'].value_counts())

# Drop any rows where feature columns contain non-numeric or missing data
df = df.dropna(subset=feature_columns)

# Convert all features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Extract features and labels
X = df[feature_columns].values
y = df['Label'].values.astype(np.float32)  # Use the mapped numeric 'Label' column

# Replace NaN values in each column with the column mean
for i in range(X.shape[1]):  # Iterate over columns
    if np.issubdtype(X[:, i].dtype, np.number):  # Ensure column is numeric
        column_mean = np.nanmean(X[:, i])  # Compute the mean of the column
        X[:, i] = np.where(np.isnan(X[:, i]), column_mean, X[:, i])  # Replace NaNs with the column mean

# Replace Infinite values with 0
X[np.isinf(X)] = 0

# Normalize the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize KNN Classifier
knn_classifier = KNeighborsClassifier(n_neighbors=5)

# Train the classifier
knn_classifier.fit(X_train, y_train)

# Make predictions
y_train_pred_knn = knn_classifier.predict(X_train)
y_test_pred_knn = knn_classifier.predict(X_test)

# Evaluate the KNN model
train_accuracy_knn = accuracy_score(y_train, y_train_pred_knn)
test_accuracy_knn = accuracy_score(y_test, y_test_pred_knn)

print(f"\nKNN Training Accuracy: {train_accuracy_knn:.4f}")
print(f"KNN Test Accuracy: {test_accuracy_knn:.4f}")

# Detailed evaluation
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred_knn))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_knn))


Label distribution:
0    3852
1     292
Name: Label, dtype: int64

KNN Training Accuracy: 0.9979
KNN Test Accuracy: 0.9928

Classification Report on Test Data:
              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00       774
         1.0       0.95      0.95      0.95        55

    accuracy                           0.99       829
   macro avg       0.97      0.97      0.97       829
weighted avg       0.99      0.99      0.99       829


Confusion Matrix:
[[771   3]
 [  3  52]]


In [4]:
#LSTM
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load the SmartThermostatDataset.csv dataset
df = pd.read_csv('SmartThermostatDataset.csv')

# Define feature columns (excluding 'Type' which is the label)
feature_columns = [
    'timestamp', 'source_ip', 'source_port', 'destination_ip', 'dst_port', 'protocol',
    'Family', 'Service', 'duration', 'source_bytes', 'destination_bytes', 'conn_state',
    'missed_bytes', 'source_packets', 'src_ip_bytes', 'destination_packets', 'dst_ip_bytes',
    'Temperature', 'Humidity', 'Status'
]

label_column = 'Type'  # 'Type' is the label

# Handle categorical features: Use LabelEncoder to convert categorical columns to numeric values
categorical_columns = ['protocol', 'Family', 'Service', 'conn_state', 'timestamp', 'source_ip', 'destination_ip', 'Status']
label_encoder = LabelEncoder()

for column in categorical_columns:
    if column in df.columns:
        df[column] = label_encoder.fit_transform(df[column].astype(str))

# Map string labels to numeric values for 'DOS' detection
df['Type'] = df['Type'].str.lower()  # Convert to lowercase

label_mapping = {
    'normal': 0,  # Non-DOS
    'mitm': 0,    # Non-DOS
    'dos': 1      # DOS
}

df['Label'] = df['Type'].map(label_mapping)

# Check for unmapped labels
if df['Label'].isna().sum() > 0:
    print("Unmapped labels found. Check your dataset!")
    print(df[df['Label'].isna()])
    raise ValueError("Unmapped labels detected.")

# Confirm the label distribution
print("Label distribution:")
print(df['Label'].value_counts())

# Drop any rows where feature columns contain non-numeric or missing data
df = df.dropna(subset=feature_columns)

# Convert all features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Extract features and labels
X = df[feature_columns].values
y = df['Label'].values.astype(np.float32)  # Use the mapped numeric 'Label' column

# Replace NaN values in each column with the column mean
for i in range(X.shape[1]):  # Iterate over columns
    if np.issubdtype(X[:, i].dtype, np.number):  # Ensure column is numeric
        column_mean = np.nanmean(X[:, i])  # Compute the mean of the column
        X[:, i] = np.where(np.isnan(X[:, i]), column_mean, X[:, i])  # Replace NaNs with the column mean

# Replace Infinite values with 0
X[np.isinf(X)] = 0

# Normalize the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Reshape data for LSTM (samples, time steps, features)
X_scaled = X_scaled.reshape((X_scaled.shape[0], 1, X_scaled.shape[1]))

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize the LSTM model
model = Sequential()

# Add LSTM layer with dropout to prevent overfitting
model.add(LSTM(units=64, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2]), return_sequences=False))
model.add(Dropout(0.2))

# Add a Dense layer with one output unit (for binary classification)
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=2)

# Make predictions
y_train_pred_lstm = (model.predict(X_train) > 0.5).astype(int)
y_test_pred_lstm = (model.predict(X_test) > 0.5).astype(int)

# Evaluate the LSTM model
train_accuracy_lstm = accuracy_score(y_train, y_train_pred_lstm)
test_accuracy_lstm = accuracy_score(y_test, y_test_pred_lstm)

print(f"\nLSTM Training Accuracy: {train_accuracy_lstm:.4f}")
print(f"LSTM Test Accuracy: {test_accuracy_lstm:.4f}")

# Detailed evaluation
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred_lstm))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_lstm))


Label distribution:
0    3852
1     292
Name: Label, dtype: int64


C:\Users\palgunitr\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:204: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/10
104/104 - 2s - 19ms/step - accuracy: 0.8989 - loss: 0.4663 - val_accuracy: 0.9337 - val_loss: 0.2800
Epoch 2/10
104/104 - 0s - 1ms/step - accuracy: 0.9285 - loss: 0.2551 - val_accuracy: 0.9337 - val_loss: 0.2325
Epoch 3/10
104/104 - 0s - 2ms/step - accuracy: 0.9285 - loss: 0.2224 - val_accuracy: 0.9337 - val_loss: 0.2050
Epoch 4/10
104/104 - 0s - 1ms/step - accuracy: 0.9291 - loss: 0.1964 - val_accuracy: 0.9337 - val_loss: 0.1850
Epoch 5/10
104/104 - 0s - 2ms/step - accuracy: 0.9321 - loss: 0.1780 - val_accuracy: 0.9385 - val_loss: 0.1687
Epoch 6/10
104/104 - 0s - 2ms/step - accuracy: 0.9345 - loss: 0.1616 - val_accuracy: 0.9397 - val_loss: 0.1530
Epoch 7/10
104/104 - 0s - 2ms/step - accuracy: 0.9381 - loss: 0.1477 - val_accuracy: 0.9421 - val_loss: 0.1377
Epoch 8/10
104/104 - 0s - 2ms/step - accuracy: 0.9427 - loss: 0.1331 - val_accuracy: 0.9445 - val_loss: 0.1233
Epoch 9/10
104/104 - 0s - 2ms/step - accuracy: 0.9469 - loss: 0.1208 - val_accuracy: 0.9505 - val_loss: 0.1119


In [5]:
# Neural Network
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Load the SmartThermostatDataset.csv dataset
df = pd.read_csv('SmartThermostatDataset.csv')

# Define feature columns (excluding 'Type' which is the label)
feature_columns = [
    'timestamp', 'source_ip', 'source_port', 'destination_ip', 'dst_port', 'protocol',
    'Family', 'Service', 'duration', 'source_bytes', 'destination_bytes', 'conn_state',
    'missed_bytes', 'source_packets', 'src_ip_bytes', 'destination_packets', 'dst_ip_bytes',
    'Temperature', 'Humidity', 'Status'
]

label_column = 'Type'  # 'Type' is the label

# Handle categorical features: Use LabelEncoder to convert categorical columns to numeric values
categorical_columns = ['protocol', 'Family', 'Service', 'conn_state', 'timestamp', 'source_ip', 'destination_ip', 'Status']
label_encoder = LabelEncoder()

for column in categorical_columns:
    if column in df.columns:
        df[column] = label_encoder.fit_transform(df[column].astype(str))

# Map string labels to numeric values for 'DOS' detection
df['Type'] = df['Type'].str.lower()  # Convert to lowercase

label_mapping = {
    'normal': 0,  # Non-DOS
    'mitm': 0,    # Non-DOS
    'dos': 1      # DOS
}

df['Label'] = df['Type'].map(label_mapping)

# Check for unmapped labels
if df['Label'].isna().sum() > 0:
    print("Unmapped labels found. Check your dataset!")
    print(df[df['Label'].isna()])
    raise ValueError("Unmapped labels detected.")

# Confirm the label distribution
print("Label distribution:")
print(df['Label'].value_counts())

# Drop any rows where feature columns contain non-numeric or missing data
df = df.dropna(subset=feature_columns)

# Convert all features to numeric
for column in feature_columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Extract features and labels
X = df[feature_columns].values
y = df['Label'].values.astype(np.float32)  # Use the mapped numeric 'Label' column

# Replace NaN values in each column with the column mean
for i in range(X.shape[1]):  # Iterate over columns
    if np.issubdtype(X[:, i].dtype, np.number):  # Ensure column is numeric
        column_mean = np.nanmean(X[:, i])  # Compute the mean of the column
        X[:, i] = np.where(np.isnan(X[:, i]), column_mean, X[:, i])  # Replace NaNs with the column mean

# Replace Infinite values with 0
X[np.isinf(X)] = 0

# Normalize the features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize the neural network model
model = Sequential()

# Add hidden layers with dropout to prevent overfitting
model.add(Dense(units=128, activation='relu', input_dim=X_train.shape[1]))  # Input layer + hidden layer
model.add(Dropout(0.2))  # Dropout layer to prevent overfitting

model.add(Dense(units=64, activation='relu'))  # Additional hidden layer
model.add(Dropout(0.2))

# Output layer with 1 unit (since it's a binary classification problem)
model.add(Dense(units=1, activation='sigmoid'))

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=2)

# Make predictions
y_train_pred_nn = (model.predict(X_train) > 0.5).astype(int)
y_test_pred_nn = (model.predict(X_test) > 0.5).astype(int)

# Evaluate the neural network model
train_accuracy_nn = accuracy_score(y_train, y_train_pred_nn)
test_accuracy_nn = accuracy_score(y_test, y_test_pred_nn)

print(f"\nNeural Network Training Accuracy: {train_accuracy_nn:.4f}")
print(f"Neural Network Test Accuracy: {test_accuracy_nn:.4f}")

# Detailed evaluation
print("\nClassification Report on Test Data:")
print(classification_report(y_test, y_test_pred_nn))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_nn))


Label distribution:
0    3852
1     292
Name: Label, dtype: int64
Epoch 1/10


C:\Users\palgunitr\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


104/104 - 1s - 11ms/step - accuracy: 0.9261 - loss: 0.2621 - val_accuracy: 0.9337 - val_loss: 0.1716
Epoch 2/10
104/104 - 0s - 1ms/step - accuracy: 0.9369 - loss: 0.1362 - val_accuracy: 0.9530 - val_loss: 0.1044
Epoch 3/10
104/104 - 0s - 1ms/step - accuracy: 0.9656 - loss: 0.0860 - val_accuracy: 0.9759 - val_loss: 0.0725
Epoch 4/10
104/104 - 0s - 1ms/step - accuracy: 0.9747 - loss: 0.0668 - val_accuracy: 0.9795 - val_loss: 0.0632
Epoch 5/10
104/104 - 0s - 1ms/step - accuracy: 0.9792 - loss: 0.0535 - val_accuracy: 0.9819 - val_loss: 0.0613
Epoch 6/10
104/104 - 0s - 1ms/step - accuracy: 0.9780 - loss: 0.0540 - val_accuracy: 0.9807 - val_loss: 0.0598
Epoch 7/10
104/104 - 0s - 1ms/step - accuracy: 0.9807 - loss: 0.0493 - val_accuracy: 0.9795 - val_loss: 0.0608
Epoch 8/10
104/104 - 0s - 1ms/step - accuracy: 0.9810 - loss: 0.0461 - val_accuracy: 0.9819 - val_loss: 0.0613
Epoch 9/10
104/104 - 0s - 1ms/step - accuracy: 0.9801 - loss: 0.0480 - val_accuracy: 0.9807 - val_loss: 0.0619
Epoch 10/10